# Week 6 MCP Exercise (`6_mcp`)

This notebook follows the Week 6 MCP exercises and keeps to the same toolchain:
1. Build your own MCP server (date/time tool)
2. Use it via OpenAI Agents SDK (`MCPServerStdio`)
3. Build a custom MCP client
4. Use native OpenAI tool calling (without Agents SDK)


## Concept Consolidation (Week 6)

- MCP server creation with `FastMCP`, exposing tools/resources over stdio.
- MCP server usage via `MCPServerStdio` inside OpenAI Agents SDK.
- Direct MCP client usage via `mcp.ClientSession` (`list_tools`, `call_tool`, `read_resource`).
- Wrapping MCP tools into OpenAI-compatible function tools for native chat completions.


## 1) Imports and Environment


In [ ]:
import os
import json
from pathlib import Path

from dotenv import load_dotenv

from agents import Agent, Runner, trace
from agents.mcp import MCPServerStdio

import mcp
from mcp.client.stdio import stdio_client
from mcp import StdioServerParameters

from openai import OpenAI

load_dotenv(override=True)

if not os.getenv('OPENAI_API_KEY') and not os.getenv('OR_API_KEY'):
    raise ValueError('Set OPENAI_API_KEY or OR_API_KEY in .env')


## 2) Confirm the Local MCP Server File


In [ ]:
server_file = Path('6_mcp/community_contributions/eddy_mmaitsimwale/date_server.py')
print('Exists:', server_file.exists())
print('Path:', server_file)


## 3) Use the MCP Server with OpenAI Agents SDK


In [ ]:
date_params = {
    'command': 'uv',
    'args': ['run', '6_mcp/community_contributions/eddy_mmaitsimwale/date_server.py']
}

async with MCPServerStdio(params=date_params, client_session_timeout_seconds=30) as server:
    tools = await server.list_tools()
    tools


In [ ]:
instructions = 'You answer date/time questions by using your MCP tools.'
request = "What is today's date in UTC?"
model = 'gpt-4.1-mini'

async with MCPServerStdio(params=date_params, client_session_timeout_seconds=30) as mcp_server:
    agent = Agent(
        name='date_assistant',
        instructions=instructions,
        model=model,
        mcp_servers=[mcp_server],
    )
    with trace('mcp_date_agent'):
        result = await Runner.run(agent, request)

print(result.final_output)


## 4) Build a Custom MCP Client


In [ ]:
params = StdioServerParameters(
    command='uv',
    args=['run', '6_mcp/community_contributions/eddy_mmaitsimwale/date_server.py'],
    env=None,
)

async def list_date_tools():
    async with stdio_client(params) as streams:
        async with mcp.ClientSession(*streams) as session:
            await session.initialize()
            tools_result = await session.list_tools()
            return tools_result.tools

async def call_date_tool(tool_name, tool_args=None):
    async with stdio_client(params) as streams:
        async with mcp.ClientSession(*streams) as session:
            await session.initialize()
            result = await session.call_tool(tool_name, tool_args or {})
            return result


In [ ]:
tools = await list_date_tools()
for t in tools:
    print(t.name, '-', t.description)

raw_result = await call_date_tool('get_current_date', {})
print(raw_result)


## 5) Native OpenAI Tool Calling (No Agents SDK)


In [ ]:
openai_api_key = os.getenv('OPENAI_API_KEY') or os.getenv('OR_API_KEY')
base_url = os.getenv('OR_BASE_URL') if os.getenv('OR_API_KEY') else None

client = OpenAI(api_key=openai_api_key, base_url=base_url)
model_name = 'gpt-4.1-mini' if not base_url else 'openai/gpt-4.1-mini'


In [ ]:
# Convert MCP tools to OpenAI function schema
openai_tools = []
for t in tools:
    schema = {**t.inputSchema, 'additionalProperties': False}
    openai_tools.append(
        {
            'type': 'function',
            'function': {
                'name': t.name,
                'description': t.description or '',
                'parameters': schema,
            },
        }
    )

messages = [
    {'role': 'system', 'content': 'You are a date assistant. Use available tools when needed.'},
    {'role': 'user', 'content': 'Please tell me current UTC date and datetime.'},
]

response = client.chat.completions.create(
    model=model_name,
    messages=messages,
    tools=openai_tools,
)

first = response.choices[0].message
print('Finish reason:', response.choices[0].finish_reason)
print('Tool calls:', first.tool_calls)


In [ ]:
# Handle any tool call(s), then send tool results back to model
if first.tool_calls:
    messages.append(first.model_dump(exclude_none=True))

    for call in first.tool_calls:
        fn_name = call.function.name
        fn_args = json.loads(call.function.arguments or '{}')
        tool_result = await call_date_tool(fn_name, fn_args)

        # collect text payload from MCP result content
        text_payload = ''
        if hasattr(tool_result, 'content') and tool_result.content:
            parts = []
            for item in tool_result.content:
                if getattr(item, 'type', None) == 'text':
                    parts.append(getattr(item, 'text', ''))
            text_payload = '\n'.join(parts).strip()

        messages.append(
            {
                'role': 'tool',
                'tool_call_id': call.id,
                'content': text_payload or str(tool_result),
            }
        )

    final = client.chat.completions.create(
        model=model_name,
        messages=messages,
    )
    print(final.choices[0].message.content)
else:
    print(first.content)


## 6) Notes

- This exercise follows the exact Week 6 direction: own MCP server + own MCP client + native OpenAI tool loop.
- If `uv run .../date_server.py` fails, make sure MCP dependencies are installed in your environment.
